In [9]:
import pandas as pd
from sklearn.model_selection import train_test_split

file_path = '../external-data/train.csv'
# Fill in the line below: preprocess test data
test_data_path = '../external-data/test.csv'

housing_df = pd.read_csv(file_path)

# read test data file using pandas
test_data = pd.read_csv(test_data_path)

# Remove rows with missing target, separate target from predictors
housing_df.dropna(axis=0, subset=['SalePrice'], inplace=True)
y = housing_df.SalePrice
housing_df.drop(['SalePrice'], axis=1, inplace=True)

# To keep things simple, we'll use only numerical predictors
X = housing_df.select_dtypes(exclude=['object'])
X_test = test_data.select_dtypes(exclude=['object'])

# Break off validation set from training data
X_train, X_valid, y_train, y_valid = train_test_split(X, y,
                                                      train_size=0.8, test_size=0.2,
                                                      random_state=0)

In [10]:
# print list of columns
housing_df.columns

Index(['Id', 'MSSubClass', 'MSZoning', 'LotFrontage', 'LotArea', 'Street',
       'Alley', 'LotShape', 'LandContour', 'Utilities', 'LotConfig',
       'LandSlope', 'Neighborhood', 'Condition1', 'Condition2', 'BldgType',
       'HouseStyle', 'OverallQual', 'OverallCond', 'YearBuilt', 'YearRemodAdd',
       'RoofStyle', 'RoofMatl', 'Exterior1st', 'Exterior2nd', 'MasVnrType',
       'MasVnrArea', 'ExterQual', 'ExterCond', 'Foundation', 'BsmtQual',
       'BsmtCond', 'BsmtExposure', 'BsmtFinType1', 'BsmtFinSF1',
       'BsmtFinType2', 'BsmtFinSF2', 'BsmtUnfSF', 'TotalBsmtSF', 'Heating',
       'HeatingQC', 'CentralAir', 'Electrical', '1stFlrSF', '2ndFlrSF',
       'LowQualFinSF', 'GrLivArea', 'BsmtFullBath', 'BsmtHalfBath', 'FullBath',
       'HalfBath', 'BedroomAbvGr', 'KitchenAbvGr', 'KitchenQual',
       'TotRmsAbvGrd', 'Functional', 'Fireplaces', 'FireplaceQu', 'GarageType',
       'GarageYrBlt', 'GarageFinish', 'GarageCars', 'GarageArea', 'GarageQual',
       'GarageCond', 'PavedDrive

In [11]:
# (rows, cols)
print(X.shape)
missing_val_count_by_column = X.isnull().sum()
print(missing_val_count_by_column[missing_val_count_by_column > 0])

(1460, 37)
LotFrontage    259
MasVnrArea       8
GarageYrBlt     81
dtype: int64


In [12]:
# Fill in the line below: How many rows are in the training data?
num_rows = 1460

# Fill in the line below: How many columns in the training data
# have missing values?
num_cols_with_missing = 3

# Fill in the line below: How many missing entries are contained in 
# all of the training data?
tot_missing = 259 + 8 + 81

In [13]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error

# Function for comparing different approaches
def score_dataset(X_train, X_valid, y_train, y_valid):
    model = RandomForestRegressor(n_estimators=100, criterion='absolute_error', random_state=0)
    model.fit(X_train, y_train)
    prediction = model.predict(X_valid)
    return mean_absolute_error(y_valid, prediction)

In [14]:
# drop columns with missing values
cols_with_missing = [col for col in X_train.columns if X_train[col].isnull().any()]

reduced_X_train = X_train.drop(cols_with_missing, axis=1)
reduced_X_valid = X_valid.drop(cols_with_missing, axis=1)

In [15]:
# check result
print("MAE from Approach 1 (Drop columns with missing values):")
print(score_dataset(reduced_X_train, reduced_X_valid, y_train, y_valid))

MAE from Approach 1 (Drop columns with missing values):
17781.28352739726


In [16]:
from sklearn.impute import SimpleImputer

# use SimpleImputer with default "mean" strategy to input data where needed
# fit_transform - Fit to data, then transform it,
# transform - input all missing values in X
my_imputer = SimpleImputer()
imputed_X_train = pd.DataFrame(my_imputer.fit_transform(X_train))
imputed_X_valid = pd.DataFrame(my_imputer.transform(X_valid))

# Imputation removed column names; put them back
imputed_X_train.columns = X_train.columns
imputed_X_valid.columns = X_valid.columns

print("MAE from Approach 2 (Imputation with mean value):")
print(score_dataset(imputed_X_train, imputed_X_valid, y_train, y_valid))

MAE from Approach 2 (Imputation with mean value):
18047.655479452056


In [17]:
# try median value
from sklearn.impute import SimpleImputer

# use SimpleImputer with strategy == "median" to input data where needed
# fit_transform - Fit to data, then transform it,
# transform - input all missing values in X
median_imputer = SimpleImputer(strategy='median')
median_X_train = pd.DataFrame(median_imputer.fit_transform(X_train))
median_X_valid = pd.DataFrame(median_imputer.transform(X_valid))

# Imputation removed column names; put them back
median_X_train.columns = X_train.columns
median_X_valid.columns = X_valid.columns

print("MAE from Approach 2 (Imputation with median value):")
print(score_dataset(median_X_train, median_X_valid, y_train, y_valid))

MAE from Approach 2 (Imputation with median value):
18025.266438356164


In [18]:
# try most_frequent value
from sklearn.impute import SimpleImputer

# use SimpleImputer with strategy == "median" to input data where needed
# fit_transform - Fit to data, then transform it,
# transform - input all missing values in X
final_imputer = SimpleImputer(strategy="most_frequent")
final_X_train = pd.DataFrame(final_imputer.fit_transform(X_train))
final_X_valid = pd.DataFrame(final_imputer.transform(X_valid))

# Imputation removed column names; put them back
final_X_train.columns = X_train.columns
final_X_valid.columns = X_valid.columns

print("MAE from Approach 2 (Imputation with most_frequent value):")
print(score_dataset(final_X_train, final_X_valid, y_train, y_valid))

MAE from Approach 2 (Imputation with most_frequent value):
18143.60705479452


In [19]:
test_model = RandomForestRegressor(n_estimators=100, criterion='absolute_error', random_state=0)
test_model.fit(median_X_train, y_train)

# use only numerical predictors
test_X = test_data.select_dtypes(exclude=['object'])

# use the exact same columns the model was trained on
final_X_test = pd.DataFrame(median_imputer.transform(test_X))

# Fill in the line below: get test predictions
preds_test = test_model.predict(final_X_test)

/opt/anaconda3/lib/python3.13/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but RandomForestRegressor was fitted with feature names
  warnings.warn(


In [20]:
# Run the code to save predictions in the format used for competition scoring

output = pd.DataFrame({'Id': test_data.Id,
                       'SalePrice': preds_test})
output.to_csv('submission-missing-values.csv', index=False)